# Phase 2: NIST 800-53 Control Mapping Engine

**Time Estimate:** 8-10 hours (theory + lab combined) | **Prerequisites:** Phase 1 completed

---

## What you'll build
A Python engine that takes raw findings from Phase 1 and maps them to NIST 800-53 controls, producing a structured compliance assessment with pass/fail status per control.

### Study cross-references
| Concept | DDIA Chapter | DVA-C02 | System Design Interview |
|---------|-------------|---------|-------------------------|
| Schema design for control mappings | Ch. 2: Data Models | DynamoDB design patterns | Ch. 6: Key-Value Store |
| Denormalization trade-offs | Ch. 2: Relational vs Document | GSI design | Ch. 5: Consistent Hashing |
| Batch processing patterns | Ch. 10: Batch Processing | Lambda batch operations | Ch. 15: Google Drive |
| Data encoding and evolution | Ch. 4: Encoding | API versioning | — |

### Documentation links
- [NIST SP 800-53 Rev 5 (full control catalog)](https://csrc.nist.gov/publications/detail/sp/800-53/rev-5/final)
- [FedRAMP Control Baselines](https://www.fedramp.gov/documents-templates/)
- [AWS Security Hub ASFF format](https://docs.aws.amazon.com/securityhub/latest/userguide/securityhub-findings-format.html)
- [boto3 Security Hub API](https://boto3.amazonaws.com/v1/documentation/api/latest/reference/services/securityhub.html)

---

## Section 1: Understanding the mapping problem

### 1.1 The three-layer mapping

Our control mapping engine connects three layers:

```
Layer 1: AWS Findings (raw data)
  │  Security Hub finding: "IAM.4 Root access key exists"
  │  Config rule: "iam-root-access-key-check" = NON_COMPLIANT
  │  IAM: Root user has access key active
  │
  ▼
Layer 2: NIST 800-53 Controls (framework)
  │  AC-2: Account Management
  │  AC-6: Least Privilege
  │  IA-2(1): Multi-Factor Authentication
  │
  ▼
Layer 3: FedRAMP Requirements (baseline)
     FedRAMP Moderate requires AC-2, AC-6, IA-2(1)
     Evidence needed: API output, configuration exports
     Assessment: PASS / FAIL / PARTIAL / NOT_ASSESSED
```

**DDIA Connection (Ch. 2 — Data Models and Query Languages):** This is a data modeling problem. We have a many-to-many relationship: one finding can map to multiple controls, and one control can have multiple findings as evidence. Kleppmann discusses how document databases handle many-to-many relationships through denormalization or application-level joins. We denormalize for query speed — each `EvidenceItem` carries a `control_ids` list so we never need a join table.

This is the same trade-off Amazon made with DynamoDB: sacrifice normalization for single-digit-millisecond reads. The cost is update anomalies (if a control ID changes, we'd need to update every evidence item), but control IDs are immutable in NIST 800-53, so denormalization is safe here.

### 1.2 Control assessment logic

For each NIST control, we determine an overall status:

```
Control AC-2 (Account Management):
  Evidence sources:
    ├── Security Hub IAM.4 → FAILED (root access key exists)
    ├── Config iam-user-mfa-enabled → NON_COMPLIANT (2 users without MFA)
    ├── Config access-keys-rotated → COMPLIANT
    └── IAM credential report → 5 users, 2 without MFA
  
  Assessment logic:
    - ANY critical/high finding FAILED → Control status = FAIL
    - ALL evidence PASSED → Control status = PASS
    - Mixed results → Control status = PARTIAL
    - No evidence collected → Control status = NOT_ASSESSED
    
  Result: AC-2 = FAIL (root access key + missing MFA)
```

**DVA-C02 Connection (Domain 1 — Development):** This assessment logic runs inside AWS Lambda. The Lambda receives a `ScanResult` event (from Phase 1), processes all evidence in-memory, and writes `ControlAssessment` items to DynamoDB. Understanding how Lambda handles JSON events and how to structure processing logic is directly tested on the exam.

---

## Section 2: Exploring the control catalog

The control catalog is the brain of your tool — it defines every NIST control we assess. Let's explore it using `src/mapper/control_catalog.py` (not redefine it).

**Important:** We import from `src/` — the same code the test suite validates. No copy-paste, no drift.

In [2]:
import sys, os
sys.path.insert(0, os.path.abspath('../..'))

from src.models import (
    EvidenceItem, ScanResult, CollectorResult, ControlAssessment,
    ControlStatus, CompliancePosture, generate_scan_id, SEVERITY_WEIGHTS, CONTROL_FAMILIES
)
from src.mapper.control_catalog import NIST_CONTROL_CATALOG, get_control, get_controls_by_family
from src.mapper.engine import ControlMappingEngine

print(f"Control catalog loaded: {len(NIST_CONTROL_CATALOG)} controls")
print(f"Control families: {len(set(c['family'] for c in NIST_CONTROL_CATALOG.values()))}")
print(f"\nFamilies:")
for fam_code in sorted(set(c['family'] for c in NIST_CONTROL_CATALOG.values())):
    controls = get_controls_by_family(fam_code)
    fam_name = CONTROL_FAMILIES.get(fam_code, fam_code)
    print(f"  {fam_code} ({fam_name}): {list(controls.keys())}")

Control catalog loaded: 25 controls
Control families: 8

Families:
  AC (Access Control): ['AC-2', 'AC-3', 'AC-6']
  AU (Audit and Accountability): ['AU-2', 'AU-3', 'AU-9', 'AU-12']
  CM (Configuration Management): ['CM-2', 'CM-3', 'CM-6', 'CM-8']
  CP (Contingency Planning): ['CP-9', 'CP-13']
  IA (Identification and Authentication): ['IA-2', 'IA-2(1)', 'IA-4', 'IA-5(1)']
  RA (Risk Assessment): ['RA-5']
  SC (System and Communications Protection): ['SC-7', 'SC-8', 'SC-13', 'SC-28']
  SI (System and Information Integrity): ['SI-2', 'SI-4', 'SI-12']


In [3]:
# Deep-dive into a single control
ac2 = get_control("AC-2")
print("AC-2: Account Management")
print(f"  Family: {ac2['family']} ({ac2['family_name']})")
print(f"  FedRAMP baselines: {ac2['fedramp_baselines']}")
print(f"  Description: {ac2['description'][:100]}...")
print(f"  Assessment criteria: {ac2['assessment_criteria'][:120]}...")

print("\n--- Compare with SC-8 (Transmission Confidentiality) ---")
sc8 = get_control("SC-8")
print(f"  FedRAMP baselines: {sc8['fedramp_baselines']}")
print(f"  Note: SC-8 is NOT in LOW baseline — only MODERATE and HIGH")
print(f"  This means a FedRAMP Low system doesn't need encrypted transit (surprising but true)")

AC-2: Account Management
  Family: AC (Access Control)
  FedRAMP baselines: ['LOW', 'MODERATE', 'HIGH']
  Description: Define and document the types of accounts allowed. Assign account managers. Establish conditions for...
  Assessment criteria: PASS: All IAM users are documented, have MFA enabled, and access keys are rotated within 90 days. FAIL: Accounts exist w...

--- Compare with SC-8 (Transmission Confidentiality) ---
  FedRAMP baselines: ['MODERATE', 'HIGH']
  Note: SC-8 is NOT in LOW baseline — only MODERATE and HIGH
  This means a FedRAMP Low system doesn't need encrypted transit (surprising but true)


## Section 3: DDIA deep dive — Document vs relational modeling (Ch. 2)

Our `NIST_CONTROL_CATALOG` is a document model — nested JSON where each control is self-contained:

```python
"AC-2": {
    "title": "Account Management",
    "family": "AC",
    "family_name": "Access Control",
    "fedramp_baselines": ["LOW", "MODERATE", "HIGH"],
    "assessment_criteria": "..."
}
```

**What would this look like in a relational model?**

```sql
-- Normalized (3NF) relational schema
CREATE TABLE control_families (
    family_code VARCHAR(4) PRIMARY KEY,   -- 'AC'
    family_name VARCHAR(100)               -- 'Access Control'
);

CREATE TABLE controls (
    control_id VARCHAR(20) PRIMARY KEY,    -- 'AC-2'
    title VARCHAR(200),
    description TEXT,
    family_code VARCHAR(4) REFERENCES control_families(family_code)
);

CREATE TABLE control_baselines (
    control_id VARCHAR(20) REFERENCES controls(control_id),
    baseline VARCHAR(20),                  -- 'MODERATE'
    PRIMARY KEY (control_id, baseline)
);
```

**Kleppmann's key insight (DDIA p.39):** "A document is usually stored as a single continuous string, encoded as JSON, XML, or a binary variant. If your application often needs to access the entire document, there is a storage locality advantage."

For our use case, we always load a control's full metadata at once — we never query "give me all controls where `fedramp_baselines` contains MODERATE" without also needing the title, description, and criteria. This makes the document model strictly better. The relational model would require 3 JOINs for what we get in one dict lookup.

**When would relational be better?** If we needed to answer "how many controls are in each baseline?" across thousands of controls, or if `assessment_criteria` were shared across controls (normalization prevents duplication). For 24 controls, the document model wins easily.

---

## Section 4: Lab — Building evidence and running the mapping engine

Now we'll create mock evidence (simulating what Phase 1's collectors produce), build a `ScanResult`, and feed it to the `ControlMappingEngine`. This is the exact same pipeline that runs in production — same classes, same engine, same logic.

### 4.1 Create realistic mock evidence

In [5]:
# Create evidence items that simulate what Phase 1 collectors would produce
# Each item has control_ids linking it to NIST controls

evidence_items = [
    # Security Hub: Root access key exists (CRITICAL)
    EvidenceItem(
        source="security_hub", finding_id="sh-iam4-001",
        title="IAM.4 Hardware MFA should be enabled for the root user",
        status="FAILED", severity="CRITICAL",
        resource_type="AWS::IAM::User",
        resource_id="arn:aws:iam::123456789012:root",
        timestamp="2024-01-15T10:00:00Z",
        remediation="Delete root access keys and enable hardware MFA",
        control_ids=["AC-2", "IA-2", "IA-2(1)"]
    ),
    # Security Hub: S3 bucket without SSL enforcement
    EvidenceItem(
        source="security_hub", finding_id="sh-s3-005",
        title="S3.5 S3 general purpose buckets should require requests to use SSL",
        status="FAILED", severity="HIGH",
        resource_type="AWS::S3::Bucket",
        resource_id="arn:aws:s3:::production-data-bucket",
        timestamp="2024-01-15T10:01:00Z",
        remediation="Add bucket policy requiring aws:SecureTransport",
        control_ids=["SC-8", "SC-13"]
    ),
    # Config: CloudTrail enabled (PASS)
    EvidenceItem(
        source="config", finding_id="cfg-cloudtrail-001",
        title="multi-region-cloudtrail-enabled: COMPLIANT",
        status="PASSED", severity="INFORMATIONAL",
        resource_type="AWS::CloudTrail::Trail",
        resource_id="arn:aws:cloudtrail:us-east-1:123456789012:trail/org-trail",
        timestamp="2024-01-15T10:02:00Z",
        control_ids=["AU-2", "AU-3", "AU-12"]
    ),
    # Config: Encrypted volumes (FAIL)
    EvidenceItem(
        source="config", finding_id="cfg-ebs-001",
        title="encrypted-volumes: NON_COMPLIANT",
        status="FAILED", severity="HIGH",
        resource_type="AWS::EC2::Volume",
        resource_id="vol-0abc123def456789",
        timestamp="2024-01-15T10:03:00Z",
        remediation="Enable EBS encryption by default in account settings",
        control_ids=["SC-13", "SC-28"]
    ),
    # IAM: Weak password policy
    EvidenceItem(
        source="iam", finding_id="iam-pwpolicy-001",
        title="Password policy does not meet NIST requirements (8 char min, no rotation)",
        status="FAILED", severity="MEDIUM",
        resource_type="AWS::IAM::AccountPasswordPolicy",
        resource_id="arn:aws:iam::123456789012:account-password-policy",
        timestamp="2024-01-15T10:04:00Z",
        remediation="Set minimum 12 chars, require complexity, 90-day rotation",
        control_ids=["IA-5(1)"]
    ),
    # Config: GuardDuty enabled (PASS)
    EvidenceItem(
        source="config", finding_id="cfg-gd-001",
        title="guardduty-enabled-centralized: COMPLIANT",
        status="PASSED", severity="INFORMATIONAL",
        resource_type="AWS::GuardDuty::Detector",
        resource_id="detector-us-east-1",
        timestamp="2024-01-15T10:05:00Z",
        control_ids=["SI-4"]
    ),
    # Config: MFA not enabled for IAM users (FAIL)
    EvidenceItem(
        source="config", finding_id="cfg-mfa-001",
        title="iam-user-mfa-enabled: NON_COMPLIANT (3 of 7 users)",
        status="FAILED", severity="HIGH",
        resource_type="AWS::IAM::User",
        resource_id="arn:aws:iam::123456789012:user/developer-jane",
        timestamp="2024-01-15T10:06:00Z",
        remediation="Enforce MFA for all IAM users with console access",
        control_ids=["IA-2(1)", "AC-2"]
    ),
    # Config: SSH restricted (PASS)
    EvidenceItem(
        source="config", finding_id="cfg-ssh-001",
        title="restricted-ssh: COMPLIANT",
        status="PASSED", severity="INFORMATIONAL",
        resource_type="AWS::EC2::SecurityGroup",
        resource_id="sg-0abc123def456789",
        timestamp="2024-01-15T10:07:00Z",
        control_ids=["SC-7", "CM-6"]
    ),
    # Config: SSM managed instances (PASS)
    EvidenceItem(
        source="config", finding_id="cfg-ssm-001",
        title="ec2-instance-managed-by-systems-manager: COMPLIANT",
        status="PASSED", severity="INFORMATIONAL",
        resource_type="AWS::EC2::Instance",
        resource_id="i-0abc123def456789",
        timestamp="2024-01-15T10:08:00Z",
        control_ids=["CM-2", "CM-8", "SI-2"]
    ),
    # Security Hub: No admin policies (PASS)
    EvidenceItem(
        source="security_hub", finding_id="sh-iam1-001",
        title="IAM.1 IAM policies should not allow full * administrative privileges",
        status="PASSED", severity="INFORMATIONAL",
        resource_type="AWS::IAM::Policy",
        resource_id="arn:aws:iam::123456789012:policy/DeveloperAccess",
        timestamp="2024-01-15T10:09:00Z",
        control_ids=["AC-6", "AC-3"]
    ),
]

print(f"Created {len(evidence_items)} evidence items")
print(f"Controls covered: {sorted(set(cid for e in evidence_items for cid in e.control_ids))}")

Created 10 evidence items
Controls covered: ['AC-2', 'AC-3', 'AC-6', 'AU-12', 'AU-2', 'AU-3', 'CM-2', 'CM-6', 'CM-8', 'IA-2', 'IA-2(1)', 'IA-5(1)', 'SC-13', 'SC-28', 'SC-7', 'SC-8', 'SI-2', 'SI-4']


### 4.2 Build a ScanResult and run the mapping engine

This is where Phase 1 output becomes Phase 2 input. In production, the `ComplianceCollector` orchestrator (from `src/collector/orchestrator.py`) builds the `ScanResult`. Here we build it manually with our mock evidence.

In [13]:
# Build a ScanResult — the same structure Phase 1 produces
scan = ScanResult(
    scan_id=generate_scan_id(),
    scan_start="2024-01-15T10:00:00Z",
    account_id="123456789012",
    region="us-east-1",
)

# Add evidence via a CollectorResult (simulating one collector)
collector_result = CollectorResult(
    source="mock_collectors",
    status="SUCCESS",
    evidence_items=evidence_items,
    raw_findings_count=len(evidence_items),
)
scan.collector_results["mock"] = collector_result
scan.finalize()

print(f"Scan ID: {scan.scan_id}")
print(f"Status: {scan.status}")
print(f"Total evidence: {len(scan.all_evidence)}")
print(f"Evidence by control: {len(scan.evidence_by_control)} controls have evidence")

# Show evidence grouping — this is the map-reduce step from DDIA Ch. 10
print("\n--- Evidence grouped by control (the 'map' step) ---")
for cid, items in sorted(scan.evidence_by_control.items()):
    statuses = [e.status for e in items]
    print(f"  {cid}: {len(items)} items — {statuses}")

Scan ID: 2026-05-05T17-40-09Z_73878cb3
Status: COMPLETED
Total evidence: 10
Evidence by control: 18 controls have evidence

--- Evidence grouped by control (the 'map' step) ---
  AC-2: 2 items — ['FAILED', 'FAILED']
  AC-3: 1 items — ['PASSED']
  AC-6: 1 items — ['PASSED']
  AU-12: 1 items — ['PASSED']
  AU-2: 1 items — ['PASSED']
  AU-3: 1 items — ['PASSED']
  CM-2: 1 items — ['PASSED']
  CM-6: 1 items — ['PASSED']
  CM-8: 1 items — ['PASSED']
  IA-2: 1 items — ['FAILED']
  IA-2(1): 2 items — ['FAILED', 'FAILED']
  IA-5(1): 1 items — ['FAILED']
  SC-13: 2 items — ['FAILED', 'FAILED']
  SC-28: 1 items — ['FAILED']
  SC-7: 1 items — ['PASSED']
  SC-8: 1 items — ['FAILED']
  SI-2: 1 items — ['PASSED']
  SI-4: 1 items — ['PASSED']


In [7]:
# Run the mapping engine — this is the 'reduce' step
engine = ControlMappingEngine()
assessments = engine.assess_all_controls(scan)

print(f"\n{'Control':<12} {'Title':<52} {'Status':<15} {'Evidence':>8} {'Priority':>8}")
print("-" * 97)
for a in sorted(assessments, key=lambda x: x.control_id):
    icon = {
        "PASS": "PASS", "FAIL": "** FAIL **",
        "PARTIAL": "~ PARTIAL", "NOT_ASSESSED": "? N/A"
    }[a.status.value]
    print(f"  {a.control_id:<10} {a.control_title:<52} {icon:<15} {a.total_findings:>8} {a.remediation_priority:>8}")


Control      Title                                                Status          Evidence Priority
-------------------------------------------------------------------------------------------------
  AC-2       Account Management                                   ** FAIL **             2       10
  AC-3       Access Enforcement                                   PASS                   1        2
  AC-6       Least Privilege                                      PASS                   1        2
  AU-12      Audit Record Generation                              PASS                   1        2
  AU-2       Event Logging                                        PASS                   1        2
  AU-3       Content of Audit Records                             PASS                   1        2
  AU-9       Protection of Audit Information                      ? N/A                  0        1
  CM-2       Baseline Configuration                               PASS                   1        2
 

In [8]:
# Generate the compliance posture — executive summary
posture = engine.generate_posture(assessments)

print("\n=== EXECUTIVE SUMMARY ===")
print(f"Total controls: {posture.total_controls}")
print(f"Applicable: {posture.applicable_controls}")
print(f"PASS: {posture.passed}  FAIL: {posture.failed}  PARTIAL: {posture.partial}  N/A: {posture.not_assessed}")
print(f"Compliance: {posture.compliance_percentage:.1f}%")
print(f"\nBy family:")
for fam, counts in sorted(posture.by_family.items()):
    print(f"  {fam}: {counts}")
if posture.top_failures:
    print(f"\nTop failures (highest priority first):")
    for f in posture.top_failures[:5]:
        print(f"  {f['control_id']}: {f['control_title']} (priority {f['remediation_priority']})")


=== EXECUTIVE SUMMARY ===
Total controls: 25
Applicable: 25
PASS: 11  FAIL: 7  PARTIAL: 0  N/A: 7
Compliance: 44.0%

By family:
  AC: {'total': 3, 'passed': 2, 'failed': 1, 'partial': 0, 'not_assessed': 0}
  AU: {'total': 4, 'passed': 3, 'failed': 0, 'partial': 0, 'not_assessed': 1}
  CM: {'total': 4, 'passed': 3, 'failed': 0, 'partial': 0, 'not_assessed': 1}
  CP: {'total': 2, 'passed': 0, 'failed': 0, 'partial': 0, 'not_assessed': 2}
  IA: {'total': 4, 'passed': 0, 'failed': 3, 'partial': 0, 'not_assessed': 1}
  RA: {'total': 1, 'passed': 0, 'failed': 0, 'partial': 0, 'not_assessed': 1}
  SC: {'total': 4, 'passed': 1, 'failed': 3, 'partial': 0, 'not_assessed': 0}
  SI: {'total': 3, 'passed': 2, 'failed': 0, 'partial': 0, 'not_assessed': 1}

Top failures (highest priority first):
  AC-2: Account Management (priority 10)
  IA-2: Identification and Authentication (Organizational Users) (priority 10)
  IA-2(1): Multi-Factor Authentication to Privileged Accounts (priority 10)
  SC-8: Tra

## Section 5: DDIA deep dive — Batch processing (Ch. 10)

What we just did is a textbook batch processing pipeline, exactly as Kleppmann describes:

```
Input (immutable)          Processing              Output (derived)
───────────��───          ────────────              ────────────────
ScanResult               ControlMappingEngine      List[ControlAssessment]
 └─ all_evidence          ├─ Group by control_id    └─ CompliancePosture
    (10 items)            ├─ Assess each control
                          └─ Aggregate posture
```

**Key DDIA principles we follow:**

1. **Immutable inputs** (p. 413): The `ScanResult` is never modified after `finalize()`. The mapping engine reads it but doesn't mutate it. This means we can re-run the engine on the same scan and get identical results — deterministic processing.

2. **Derived datasets** (p. 419): `ControlAssessment` and `CompliancePosture` are derived from `ScanResult`. We could throw them away and recompute them anytime. This is the same principle behind materialized views in databases.

3. **Sort-merge joins** (p. 403): Our `evidence_by_control` property pre-sorts evidence by control ID. The mapping engine then iterates through controls in catalog order and looks up matching evidence — conceptually the same as a sort-merge join between the catalog and the evidence stream.

**Why this matters for the DVA-C02 exam:** Lambda functions that process batches of records (from DynamoDB Streams, SQS, or Kinesis) follow exactly this pattern. The exam tests whether you understand idempotency, error handling in batch processing, and how to handle partial failures.

---

---
## Section 6: Exercise 2.1 — Investigating SC-7 (Boundary Protection)

### What You're About to Do — And Why It Matters

This exercise is not just "find a control and print its status." It is a **debugging and reasoning exercise** — the exact skill you will use daily as a cloud security engineer. You will:

1. Pull a specific control's assessment out of a list
2. Inspect every piece of evidence that fed into it
3. Discover a mismatch between what the exercise description *claims* and what the data *shows*
4. Explain why the engine made the decision it did

That last step — **explaining a system's behavior from its data** — is the difference between someone who runs tools and someone who builds them.

---

### Plain English: What Is the SC Family?

The SC (System and Communications Protection) family covers everything about how data travels and where it lives. There are four SC controls in our catalog. They sound similar but are distinct requirements:

```
SC-7  Boundary Protection
      "Who is allowed to knock on the door?"
      → Security group rules, firewall inbound/outbound rules, VPC NACLs
      → AWS checks: restricted-ssh, no public S3, no unrestricted ingress
      → Think: the LOCK on the door

SC-8  Transmission Confidentiality and Integrity
      "Is the conversation encrypted while in transit?"
      → TLS/HTTPS enforcement, no HTTP-only endpoints
      → AWS checks: S3 deny-non-SSL bucket policy, ALB HTTPS redirect
      → Think: the ENVELOPE around the message

SC-13 Cryptographic Protection
      "Are we using approved encryption algorithms?"
      → FIPS 140-2 validated cryptography, KMS key management
      → AWS checks: EBS encryption, RDS encryption, S3 SSE
      → Think: the QUALITY of the lock

SC-28 Protection of Information at Rest
      "Is data encrypted while sitting in storage?"
      → Database encryption, disk encryption, backup encryption
      → AWS checks: EBS volumes encrypted, RDS encrypted, DynamoDB encrypted
      → Think: the SAFE where data sleeps
```

**Why this matters for real work:** Auditors ask "is data encrypted in transit?" (SC-8) and "is data encrypted at rest?" (SC-28) as separate questions. Saying "yes we have encryption" without specifying which type fails the audit. Our engine maps findings to the *correct* control — not just any SC control.

---

### The Key Insight Before You Run the Code

The exercise description says: *"SC-7 should have mixed evidence — SSH restricted = PASS, but the S3 SSL finding maps here too."*

**This description is subtly wrong.** Here is what actually happened when we built the evidence items:

```python
# cfg-ssh-001: restricted SSH (PASS)
EvidenceItem(
    ...
    control_ids=["SC-7", "CM-6"]   ← maps to SC-7 ✓
)

# sh-s3-005: S3 buckets require SSL (FAIL)
EvidenceItem(
    ...
    control_ids=["SC-8", "SC-13"]  ← maps to SC-8, NOT SC-7
)
```

The S3 SSL finding maps to **SC-8** (Transmission Confidentiality) — not SC-7 (Boundary Protection). This is *correct* NIST mapping. Requiring SSL on an S3 bucket is about encrypting data in transit, not about who can reach the bucket (boundary). So SC-7 ends up with only one piece of evidence (PASSED), and its status is **PASS**.

If you expected PARTIAL, you were thinking of SC-7 as a general "security" bucket. The system is more precise than that.

**What PARTIAL would actually look like:**
```python
# To make SC-7 PARTIAL, you'd need a finding like this:
EvidenceItem(
    finding_id="cfg-public-bucket-001",
    title="s3-bucket-public-read-prohibited: NON_COMPLIANT",
    status="FAILED",
    control_ids=["SC-7"]   ← now SC-7 has PASS + FAIL → PARTIAL
)
```

---

### Python Patterns You Will Use in This Code

Before running the cells, internalize these two patterns. They appear constantly:

**Pattern 1 — `next()` with a generator expression (safe lookup):**
```python
# Plain English: "Find the first assessment where control_id is 'SC-7'.
# If none exists, return None instead of crashing."
sc7 = next((a for a in assessments if a.control_id == "SC-7"), None)

# Breakdown:
#   (a for a in assessments if a.control_id == "SC-7")
#         ↑ generator: streams items one at a time, doesn't load all into memory
#   next(..., None)
#         ↑ takes the first item, or returns None if the generator is empty
#   without the None default: raises StopIteration if not found (a crash)

# When to use it: searching a list for ONE specific item by a field value.
# When NOT to use it: when you expect multiple matches (use a list comprehension instead).
```

**Pattern 2 — `dict.get(key, default)` for safe dict access:**
```python
sc7_evidence = scan.evidence_by_control.get("SC-7", [])

# Plain English: "Give me the list of evidence for SC-7.
# If SC-7 has no evidence, give me an empty list instead of a KeyError."
# 
# Without .get(): scan.evidence_by_control["SC-7"] → KeyError if SC-7 has no evidence
# With .get():    returns [] safely → the for loop just doesn't execute
```

**Pattern 3 — f-string alignment with `:<N`:**
```python
print(f"  {item.status:<10}")
# :<10 = left-align in a field 10 characters wide
# :>10 = right-align in a field 10 characters wide
# Used to make columns line up in terminal output
```

---

### Assessment Logic — How the Engine Decides PASS / FAIL / PARTIAL

The `ControlMappingEngine` uses a waterfall decision:

```
For each control in the catalog:
  1. Collect all evidence items where control_id is in item.control_ids
  2. If no evidence → NOT_ASSESSED (we couldn't check this)
  3. If any evidence status == "FAILED" AND any status == "PASSED" → PARTIAL
  4. If ALL evidence == "FAILED" → FAIL
  5. If ALL evidence == "PASSED" → PASS

Severity of the highest-severity FAILED finding becomes the control's severity.
Priority = f(status, severity, failed_finding_count)
```

This logic lives in `src/mapper/engine.py`. Open it and trace through `assess_control()` — you'll see exactly these conditions. Reading the source after understanding the theory is how you move from using a tool to owning it.

---

### Relevant Reading (Highly Specific)

| Topic | Resource |
|---|---|
| SC-7 full control text | [NIST SP 800-53 Rev 5 SC-7](https://csrc.nist.gov/Projects/risk-management/sp800-53-controls/release-search#!/controls?version=5.1&family=SC) |
| FedRAMP SC-7 implementation requirements | [FedRAMP High Baseline SC-7](https://www.fedramp.gov/documents-templates/) |
| AWS Config rules that map to SC-7 | [AWS Config Managed Rules — restricted-ssh](https://docs.aws.amazon.com/config/latest/developerguide/restricted-ssh.html) |
| AWS Security Hub finding format (ASFF) | [ASFF — ProductFields and Compliance](https://docs.aws.amazon.com/securityhub/latest/userguide/securityhub-findings-format-attributes.html) |
| Python `next()` with generators | [Python docs — Built-in Functions: next()](https://docs.python.org/3/library/functions.html#next) |
| Python `dict.get()` | [Python docs — dict.get(key, default)](https://docs.python.org/3/library/stdtypes.html#dict.get) |
| DDIA Ch. 2 — many-to-many in document models | Designing Data-Intensive Applications p. 36–42 |

---


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# STEP 1 OF 4: Pull SC-7's ControlAssessment from the list
# ═══════════════════════════════════════════════════════════════════════════════
#
# `assessments` is a List[ControlAssessment] — 25 items, one per catalog control.
# We need the single item where control_id == "SC-7".
#
# BOILERPLATE: This 1-liner appears every time you need one specific item from a list.
# ───────────────────────────────────────────────────────────────────────────────
sc7 = next(
    (a for a in assessments if a.control_id == "SC-7"),
    None   # ← default: return None if SC-7 isn't found (safe — no crash)
)

# Guard: always check before using. If None → something is wrong with the catalog.
if sc7 is None:
    print("ERROR: SC-7 not found in assessments. Check that the catalog loaded correctly.")
else:
    # ── What ControlAssessment looks like ─────────────────────────────────────
    # These are the top-level fields computed by ControlMappingEngine.assess_control():
    print(f"Control ID:       {sc7.control_id}")          # 'SC-7'
    print(f"Title:            {sc7.control_title}")       # human name
    print(f"Family:           {sc7.control_family}")      # 'SC'
    print(f"")
    print(f"STATUS:           {sc7.status.value}")        # PASS / FAIL / PARTIAL / NOT_ASSESSED
    print(f"  Passed:         {sc7.passed_findings}")     # count of PASSED evidence items
    print(f"  Failed:         {sc7.failed_findings}")     # count of FAILED evidence items
    print(f"  Total:          {sc7.total_findings}")      # passed + failed
    print(f"")
    print(f"Highest severity: {sc7.highest_severity}")    # severity of the worst failed finding
    print(f"Priority:         {sc7.remediation_priority}/10")  # 10=fix now, 1=low urgency

    # ── What evidence went in? ─────────────────────────────────────────────────
    # sc7.evidence is the list of EvidenceItem objects that reference SC-7
    print(f"\nEvidence ({len(sc7.evidence)} item(s)):")
    for item in sc7.evidence:
        icon = "✓" if item.status == "PASSED" else "✗"
        print(f"  {icon} [{item.status}]  {item.title}")
        print(f"       finding_id:   {item.finding_id}")
        print(f"       source:       {item.source}")
        print(f"       resource_id:  {item.resource_id}")
        print(f"       control_ids:  {item.control_ids}  ← ALL controls this finding maps to")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# STEP 2 OF 4: Verify the claim — did the S3 SSL finding actually map to SC-7?
# ═══════════════════════════════════════════════════════════════════════════════
#
# The exercise description claims the S3 SSL finding maps to SC-7.
# Let's verify by looking at the raw EvidenceItem directly.
#
# `evidence_items` is the original flat list we built in Section 4.
# We search it for the specific finding by its finding_id.
# ───────────────────────────────────────────────────────────────────────────────

s3_ssl_finding = next(
    (e for e in evidence_items if e.finding_id == "sh-s3-005"),
    None
)

print("=== STEP 2: Verify the S3 SSL finding's control_ids ===\n")
print(f"Finding:     {s3_ssl_finding.title}")
print(f"Status:      {s3_ssl_finding.status}")
print(f"Control IDs: {s3_ssl_finding.control_ids}")
print()
print("What this means:")
print("  SC-8  = Transmission Confidentiality (requiring SSL = encrypting the channel)")
print("  SC-13 = Cryptographic Protection (approved encryption in use)")
print()
print("  'SC-7' is NOT in this list.")
print("  SC-7 is Boundary Protection — it's about who can CONNECT, not HOW data is encrypted.")
print()

# Now show what IS mapped to SC-7 from scan.evidence_by_control
print("=== What IS in scan.evidence_by_control['SC-7']? ===\n")

# scan.evidence_by_control is a dict built by ScanResult.finalize()
# Keys = control_id strings, Values = List[EvidenceItem]
# It is built by iterating all evidence and grouping by each item's control_ids
sc7_evidence = scan.evidence_by_control.get("SC-7", [])

print(f"SC-7 has {len(sc7_evidence)} evidence item(s):\n")
for item in sc7_evidence:
    icon = "✓" if item.status == "PASSED" else "✗"
    print(f"  {icon}  finding_id:   {item.finding_id}")
    print(f"     title:        {item.title}")
    print(f"     status:       {item.status}")
    print(f"     severity:     {item.severity}")
    print(f"     control_ids:  {item.control_ids}")
    print()

print("─" * 60)
print("CONCLUSION:")
print("  SC-7 has 1 evidence item → restricted-ssh → PASSED")
print("  With only PASSED evidence, the engine returns PASS.")
print("  There is no FAILED evidence, so PARTIAL is impossible.")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# STEP 3 OF 4: Compare all four SC controls side by side
# ═══════════════════════════════════════════════════════════════════════════════
#
# Now we understand WHY SC-7 got PASS. Let's see the full picture for the SC family
# to understand how the S3 SSL finding *did* land (on SC-8 and SC-13).
#
# Pattern: iterate a known list of IDs, pull each assessment, format a table.
# ───────────────────────────────────────────────────────────────────────────────

SC_FAMILY_CONTROLS = ["SC-7", "SC-8", "SC-13", "SC-28"]

# Build a lookup dict for O(1) access: {control_id: ControlAssessment}
# Without this we'd call next() four times; with it we call it once.
assessment_by_id = {a.control_id: a for a in assessments}
#                   ↑ dict comprehension: same as a for-loop building a dict

print(f"{'Control':<8} {'Title':<42} {'Status':<14} {'Passed':>6} {'Failed':>6} {'Evidence summary'}")
print("─" * 100)

for cid in SC_FAMILY_CONTROLS:
    a = assessment_by_id[cid]   # O(1) lookup — no search needed

    # Get the evidence titles for this control (truncated to fit the line)
    evidence_for_control = scan.evidence_by_control.get(cid, [])
    evidence_summary = " | ".join(
        f"{'✓' if e.status == 'PASSED' else '✗'} {e.finding_id}"
        for e in evidence_for_control
    ) or "(none)"

    print(f"{cid:<8} {a.control_title[:41]:<42} {a.status.value:<14} {a.passed_findings:>6} {a.failed_findings:>6}  {evidence_summary}")

print()
print("Reading the table:")
print("  SC-7:  1 PASSED finding → PASS  (boundary checks are good)")
print("  SC-8:  1 FAILED finding → FAIL  (the S3 SSL finding landed HERE, not SC-7)")
print("  SC-13: 2 FAILED findings → FAIL (EBS unencrypted + S3 no-SSL both map here)")
print("  SC-28: 1 FAILED finding → FAIL  (EBS unencrypted maps here)")
print()
print("The S3 SSL finding (sh-s3-005) maps to SC-8 AND SC-13 simultaneously.")
print("One EvidenceItem can count as evidence for multiple controls at once.")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# STEP 4 OF 4: Prove PARTIAL — add a FAILED finding to SC-7 and re-run
# ═══════════════════════════════════════════════════════════════════════════════
#
# Edge case exercise: what does PARTIAL actually look like?
# We simulate it by adding a new FAILED EvidenceItem that maps to SC-7.
#
# This is the scientific method: form a hypothesis, change one variable, observe.
# "If I add a FAILED finding to SC-7, the engine should return PARTIAL."
# ───────────────────────────────────────────────────────────────────────────────

# New finding: a public S3 bucket (a real SC-7 boundary violation)
public_bucket_finding = EvidenceItem(
    source="config",
    finding_id="cfg-s3-public-001",
    title="s3-bucket-public-read-prohibited: NON_COMPLIANT — logs-archive bucket is public",
    status="FAILED",
    severity="HIGH",
    resource_type="AWS::S3::Bucket",
    resource_id="arn:aws:s3:::logs-archive",
    timestamp="2024-01-15T10:10:00Z",
    remediation="Enable S3 Block Public Access on the logs-archive bucket",
    control_ids=["SC-7", "AC-3"],   # ← SC-7 now has both a PASS and a FAIL
)

# Rebuild the scan with the new evidence added
all_evidence_with_partial = evidence_items + [public_bucket_finding]

scan_partial = ScanResult(
    scan_id=generate_scan_id(),
    scan_start="2024-01-15T10:00:00Z",
    account_id="123456789012",
    region="us-east-1",
)
scan_partial.collector_results["mock"] = CollectorResult(
    source="mock",
    status="SUCCESS",
    evidence_items=all_evidence_with_partial,
    raw_findings_count=len(all_evidence_with_partial),
)
scan_partial.finalize()

# Re-run the engine on the new scan
assessments_partial = engine.assess_all_controls(scan_partial)

# Pull SC-7 from the new results
sc7_partial = next(a for a in assessments_partial if a.control_id == "SC-7")

print("=== BEFORE (original scan) ===")
print(f"  SC-7 evidence count: {sc7.total_findings}")
print(f"  SC-7 status:         {sc7.status.value}")
print()
print("=== AFTER (added public bucket finding) ===")
print(f"  SC-7 evidence count: {sc7_partial.total_findings}")
print(f"  SC-7 passed:         {sc7_partial.passed_findings}")
print(f"  SC-7 failed:         {sc7_partial.failed_findings}")
print(f"  SC-7 status:         {sc7_partial.status.value}")   # ← should now be PARTIAL
print(f"  SC-7 severity:       {sc7_partial.highest_severity}")
print(f"  SC-7 priority:       {sc7_partial.remediation_priority}/10")
print()
print("Evidence in the new SC-7:")
for item in scan_partial.evidence_by_control.get("SC-7", []):
    icon = "✓" if item.status == "PASSED" else "✗"
    print(f"  {icon} [{item.status:<10}] {item.title}")
print()
print("─" * 60)
print("KEY TAKEAWAYS:")
print("  1. PARTIAL = at least one PASS and at least one FAIL for the same control")
print("  2. Adding one FAILED finding changed SC-7 from PASS → PARTIAL")
print("  3. The priority jumped from 2 to ~6 because of the HIGH severity failure")
print("  4. The control_ids list on each EvidenceItem is the JOIN KEY between")
print("     evidence and controls — it's what makes the many-to-many mapping work")

---
### Exercise 2.1 Debrief — What a Senior Engineer Would Take From This

You just did three things that senior engineers do every day:

**1. Challenged an assumption with data.**
The exercise said SC-7 should have mixed evidence. You didn't assume it was right — you ran the code and inspected the raw `control_ids` on the finding. The finding mapped to SC-8, not SC-7. **Evidence beats assumption.** This is how you debug production issues: don't assume you know what the system did, read what it actually did.

**2. Traced a value through multiple layers.**
`sh-s3-005` started as a raw `EvidenceItem` → got grouped into `scan.evidence_by_control["SC-8"]` → got pulled by the engine when assessing SC-8 → contributed a FAILED status to SC-8's ControlAssessment. You now understand every layer that value passed through. This is called **data lineage** — knowing where a value came from and how it was transformed.

**3. Proved a hypothesis by changing one variable.**
To understand PARTIAL, you didn't just read the definition — you created a controlled experiment: add exactly one FAILED finding to SC-7, re-run, observe the status change. This is how you validate that your mental model of a system is correct.

---

### The Patterns That Will Repeat Forever

```python
# Pattern: safe single-item search — use this anytime you need ONE item from a list
item = next((x for x in my_list if x.some_field == target), None)

# Pattern: safe dict access — use this anytime you're not sure a key exists
items = my_dict.get(key, [])   # returns empty list, not KeyError

# Pattern: dict comprehension for fast lookup — use when you'll search many times
lookup = {item.id: item for item in my_list}   # build once, use O(1) many times
result = lookup["SC-7"]                         # instant, no search

# Pattern: controlled experiment — change one thing, observe the result
# This is unit testing applied to your own understanding of the system.
```

---

### What to Do Next

1. **Open `src/mapper/engine.py`** and find the method that computes the status (look for the PARTIAL logic). Confirm it matches the waterfall description above.
2. **Open `src/collector/config_collector.py`** and find the `RULE_TO_NIST_MAP`. Notice that `restricted-ssh` maps to `["AC-3"]` — not `["SC-7"]`. This means the SSH finding in our mock data would NOT appear in a real Config scan for SC-7. Our mock data chose SC-7 deliberately for teaching purposes.
3. **Try Exercise 2.2** — add evidence for three NOT_ASSESSED controls and prove that compliance percentage rises.

---

### Exercise 2.2: Add evidence for untested controls

Several controls have NOT_ASSESSED status because our mock data doesn't cover them. Pick 2-3 controls that show as NOT_ASSESSED (like AU-9, CM-3, IA-4) and create `EvidenceItem` objects that would cause them to become PASS or FAIL. Re-run the engine and observe the change.

In [15]:
# Exercise 2.2: YOUR TURN
# Add evidence for controls that currently show NOT_ASSESSED
# Then re-run the engine and compare

new_evidence = [
    # Example: Add evidence for AU-9 (Protection of Audit Information)
    EvidenceItem(
        source="config", finding_id="cfg-logval-001",
        title="cloud-trail-log-file-validation-enabled: COMPLIANT",
        status="PASSED", severity="INFORMATIONAL",
        resource_type="AWS::CloudTrail::Trail",
        resource_id="arn:aws:cloudtrail:us-east-1:trail/org-trail",
        timestamp="2024-01-15T11:00:00Z",
        control_ids=["AU-9"]
    ),
    # YOUR TURN: Add evidence for CM-3 and IA-4
    # ...
]

# Rebuild scan with additional evidence
all_evidence = evidence_items + new_evidence
scan2 = ScanResult(scan_id=generate_scan_id(), scan_start="2024-01-15T11:00:00Z",
                   account_id="123456789012", region="us-east-1")
cr2 = CollectorResult(source="mock", status="SUCCESS",
                      evidence_items=all_evidence, raw_findings_count=len(all_evidence))
scan2.collector_results["mock"] = cr2
scan2.finalize()

assessments2 = engine.assess_all_controls(scan2)
posture2 = engine.generate_posture(assessments2)

print(f"Before: {posture.compliance_percentage:.1f}% ({posture.not_assessed} not assessed)")
print(f"After:  {posture2.compliance_percentage:.1f}% ({posture2.not_assessed} not assessed)")

Before: 44.0% (7 not assessed)
After:  48.0% (6 not assessed)


### Exercise 2.3: DVA-C02 practice — DynamoDB schema design

Design the DynamoDB schema for storing control assessments. Write the `put_item` and `query` calls for:
1. Storing an assessment (PK = `CTRL#AC-2`, SK = `SCAN#2024-01-15T10-00-00Z`)
2. Querying all FAILED controls for a scan (needs a GSI)
3. Querying the history of a specific control across all scans

**Hint:** Think single-table design. Review the DynamoDB module in `terraform/modules/dynamodb/main.tf` to see the GSIs we've already defined.

In [ ]:
# Exercise 2.3: YOUR TURN — DynamoDB schema design
# This is a pen-and-paper exercise, but let's model the items in Python

# Item 1: Store a ControlAssessment as a DynamoDB item
dynamodb_item = {
    "PK": "CTRL#AC-2",
    "SK": f"SCAN#{scan.scan_id}",
    "GSI1PK": f"SCAN#{scan.scan_id}",
    "GSI1SK": "CTRL#AC-2",
    "control_title": "Account Management",
    "status": "FAIL",
    "family": "AC",
    "failed_findings": 2,
    "passed_findings": 0,
    "highest_severity": "CRITICAL",
    "remediation_priority": 10,
    "ttl": 1737000000,  # Expire after 1 year
}

print("DynamoDB item for AC-2:")
for k, v in dynamodb_item.items():
    print(f"  {k}: {v}")

# Question: What query would get ALL failed controls for this scan?
# Answer: Query GSI1 where GSI1PK = 'SCAN#...' and filter status = 'FAIL'
# But filtering is expensive! Better: add GSI2PK = 'STATUS#FAIL', GSI2SK = 'SCAN#...'
print("\nBetter design: Add GSI2 for status-based queries")
print("  GSI2PK = 'STATUS#FAIL'")
print("  GSI2SK = 'SCAN#2024-01-15T10-00-00Z_abc123#CTRL#AC-2'")
print("  This lets you query all failures for a scan in one call")

### Exercise 2.4: DDIA exercise — Document vs relational (Ch. 2)

Our `NIST_CONTROL_CATALOG` uses a document model (nested JSON). We discussed the relational alternative in Section 3.

**Your task:** Open `src/mapper/control_catalog.py` and `src/models.py`. Answer:
1. What fields in `ControlAssessment` come from the catalog vs computed at runtime?
2. If we added a `remediation_steps: List[str]` field to each control, would that require a schema migration of stored assessments? Why or why not? (Hint: DDIA Ch. 4 — schema evolution)
3. Our `EvidenceItem.control_ids` is a denormalized list. When would this cause problems?

---

## Summary

**What you built:**
- Loaded the NIST control catalog from `src/mapper/control_catalog.py` (24 controls, 8 families)
- Created mock evidence using `src/models.EvidenceItem`
- Built a `ScanResult` and ran the `ControlMappingEngine`
- Generated a `CompliancePosture` executive summary
- Explored DDIA batch processing concepts in depth

**Key classes used (all from `src/`):**
- `EvidenceItem` → atomic unit of evidence
- `ScanResult` → complete scan output (Phase 1 produces this)
- `ControlMappingEngine` → maps evidence to controls (this phase)
- `ControlAssessment` → one per control per scan
- `CompliancePosture` → aggregate summary

**Next: Phase 3** → Take these `ControlAssessment` objects and generate audit-ready PDF reports.